# Notebook 0 — Flight pre-check

Run this **once** before Notebooks 1–3. It verifies the environment and **downloads the workshop image subset** so later notebooks can stay short and focused on the science.

| Check | Why it matters |
|-------|----------------|
| Python packages | Torch / transformers / Hydra must import |
| `.env` + Vector key | API judge + Nano Banana 2 use `OPENAI_API_KEY` (`vp_…`) |
| Vector proxy ping | Confirms network + key against `proxy.vectorinstitute.ai` |
| GPU / CUDA | Picks `cpu` vs `gpu_l4` vs `gpu_l4x2` |
| **Sample images** | Mapillary toy set (fetched here with a progress bar if missing) |
| Model IDs | What this hardware profile will load (cached vs download-on-first-use) |

**Does not** download Klein or run diffusion — that stays in Notebook 1.

Next: [Notebook 1.5](01.5_method_comparison.ipynb) (optional method bake-off) → [Notebook 1](01_sample_data_generation.ipynb).

---
## 0. Setup

From the **repo root** (once per machine):

```bash
uv sync --dev --group edge-case-image-generation

cp implementations/edge_case_image_generation/.env.example \
   implementations/edge_case_image_generation/.env
# paste your Vector proxy key into OPENAI_API_KEY
# optional: HF_TOKEN=…  (or use interactive login in the data cell below)
```

Select the project kernel, then run the cells below. Sample images are pulled in **§2** (no separate extract script required). The CLI twin still exists if you prefer: `uv run python scripts/extract_mapillary_toy.py`.

In [ ]:
import sys
from pathlib import Path


def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    search = [here, *here.parents]
    for base in list(search):
        nested = base / "implementations" / "edge_case_image_generation"
        if nested.is_dir():
            search.append(nested)
    for base in search:
        if (base / "src" / "edgecase_synthesis").is_dir() and (base / "configs").is_dir():
            return base
    raise FileNotFoundError("Could not find edge_case_image_generation root — cd to the repo or notebooks folder")


PROJECT_ROOT = _find_project_root()
src = str(PROJECT_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from edgecase_synthesis.config import load_env


env_path = load_env(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print(".env         =", env_path or "(missing — copy .env.example)")

---
## 1. Knobs

- `DATASET` — Hydra dataset package under `configs/datasets/`.
- `PING_PROXY` — set `False` if you are offline and only want local checks.
- `HARDWARE_OVERRIDE` — leave `None` to auto-recommend from GPUs; or force `"cpu"` / `"gpu_l4"` / `"gpu_l4x2"`.
- `FORCE_REEXTRACT` — wipe and rebuild the toy image subset.
- `MIN_SAMPLES` — treat the cache as incomplete below this count.

In [ ]:
DATASET = "mapillary_vistas"
PING_PROXY = True
HARDWARE_OVERRIDE = None  # e.g. "gpu_l4" to force
FORCE_REEXTRACT = False
MIN_SAMPLES = 1

---
## 2. Workshop data (Mapillary toy subset)

Checks `data/.../samples/`. If empty (or `FORCE_REEXTRACT`), extracts a small tagged subset via authenticated zip **Range GETs** — not the full ~29 GB archive. Needs HF access (`.env` `HF_TOKEN` or interactive login).

In [ ]:
import os

from edgecase_synthesis.preflight import ensure_workshop_data
from huggingface_hub import get_token, login


if not (get_token() or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")):
    print("No HF token in env/cache — starting interactive login…")
    login()

sample_paths = ensure_workshop_data(
    PROJECT_ROOT,
    dataset_name=DATASET,
    clean=FORCE_REEXTRACT,
    min_images=MIN_SAMPLES,
)
print(f"Ready: {len(sample_paths)} sample image(s)")

---
## 3. Run preflight

Rows marked **XX** must be fixed before Notebook 1. **!!** warnings are usually OK for a smoke test but will slow you down or block gated downloads.

In [ ]:
from edgecase_synthesis.preflight import run_preflight


report = run_preflight(
    PROJECT_ROOT,
    dataset_name=DATASET,
    ping_proxy=PING_PROXY,
    hardware_override=HARDWARE_OVERRIDE,
)
report.print_table()

HARDWARE = report.recommended_hardware
print(f"\nCopy into later notebooks:  HARDWARE = {HARDWARE!r}  DATASET = {DATASET!r}")

---
## 4. What each later notebook expects

| Notebook | Needs from this preflight |
|----------|---------------------------|
| **1.5** Method comparison | Same key + hardware; optional Nano Banana uses the **same** Vector proxy |
| **1** Single-image loop | Samples + GPU (or patience on CPU) + judge API |
| **2** Batch synth | Dual-GPU helps; CLI: `scripts/run_nb2_batch.py` |
| **3** Detector train/eval | Needs NB2 export under `outputs/<dataset>/nb2/` |

If preflight is **READY**, jump to [Notebook 1](01_sample_data_generation.ipynb) (or [1.5](01.5_method_comparison.ipynb) if you want to compare edit methods first).

In [ ]:
assert report.ready, (
    "Preflight failed — fix the XX rows above before continuing. Re-run after editing .env or re-running the data cell."
)
print("All required checks passed. You're clear for Notebook 1.")